# Auditoria manual DENUE: registros en `revisar`

Este notebook apoya la auditoria manual de los 182 registros DENUE que quedaron en categoria `revisar`. No rehace el pipeline completo ni sobreescribe las plantillas originales: carga las tres plantillas existentes, agrega columnas de apoyo, sugiere una prioridad de revision y exporta versiones enriquecidas para trabajar con mas contexto.

Las categorias sugeridas son una ayuda operativa. La decision final debe quedar en `categoria_auditada` y documentarse en `notas_auditoria`. La distancia a hidrografia se usa como priorizacion espacial, no como prueba de contaminacion directa.

## 0. Configuracion

Ajusta `HYDRO_THRESHOLD_M` si quieres cambiar el umbral usado para marcar registros cercanos a hidrografia.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent

if str(ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(ROOT / 'scripts'))

from audit_denue_helpers import (
    REGION_METADATA,
    REQUIRED_AUDIT_COLUMNS,
    audit_status_counts,
    export_audit_outputs,
    load_audit_templates,
    make_audit_summary,
    make_initial_summary,
    prepare_audit_table,
    validate_audit_outputs,
    write_markdown_report,
)

TABLES = ROOT / 'outputs' / 'tables'
MAPS = ROOT / 'outputs' / 'maps'
ENRICHED_DIR = TABLES / 'auditoria_enriquecida'

HYDRO_THRESHOLD_M = 250
EXPECTED_TOTAL_REVISAR = 182

AUDIT_FILES = {
    'huejotzingo': TABLES / 'auditoria_revisar_huejotzingo.csv',
    'xalmimilulco': TABLES / 'auditoria_revisar_xalmimilulco.csv',
    'san_martin': TABLES / 'auditoria_revisar_san_martin.csv',
}

for region_id, path in AUDIT_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f'No existe la plantilla esperada para {region_id}: {path}')

display(Markdown(f'Proyecto: `{ROOT}`'))
display(Markdown(f'Umbral de cercania a hidrografia: **{HYDRO_THRESHOLD_M} m**'))

## 1. Cargar plantillas y resumen inicial

Se leen las tres plantillas originales sin modificarlas.

In [ ]:
templates = load_audit_templates(AUDIT_FILES)
initial_summary = make_initial_summary(templates)

display(Markdown('### Resumen por localidad/municipio'))
display(initial_summary.drop(columns=['columnas_disponibles']))

for region_id, df in templates.items():
    label = REGION_METADATA[region_id]['label']
    display(Markdown(f'### Columnas disponibles: {label}'))
    display(pd.DataFrame({'columna': df.columns}))
    display(Markdown(f'### Estado de auditoria: {label}'))
    display(audit_status_counts(df))

## 2. Enriquecer tablas de auditoria

Este paso agrega columnas auxiliares si no existen: queries de busqueda, keywords detectadas, criterio sugerido, categoria sugerida, prioridad de revision y campos para documentar fuente/decision. Las plantillas originales permanecen intactas.

In [ ]:
enriched_tables = {
    region_id: prepare_audit_table(
        df,
        region_id=region_id,
        hydro_threshold_m=HYDRO_THRESHOLD_M,
    )
    for region_id, df in templates.items()
}

all_enriched = pd.concat(enriched_tables.values(), ignore_index=True)

created_or_verified = [col for col in REQUIRED_AUDIT_COLUMNS if col in all_enriched.columns]
display(Markdown('### Columnas de auditoria verificadas/agregadas'))
display(pd.DataFrame({'columna': created_or_verified}))

display(Markdown(f'### Registros enriquecidos: **{len(all_enriched):,}**'))
display(all_enriched.head(10))

## 3. Resumen de auditoria sugerida

Este resumen ayuda a dimensionar la revision. No sustituye la decision manual.

In [ ]:
summary = make_audit_summary(enriched_tables, hydro_threshold_m=HYDRO_THRESHOLD_M)

display(Markdown(f"### Total de registros `revisar` procesados: **{summary['total_registros']:,}**"))
display(Markdown(f"- Posible alta relevancia ambiental: **{summary['posible_alta_relevancia']:,}**"))
display(Markdown(f"- Cercanos a hidrografia (<= {HYDRO_THRESHOLD_M} m): **{summary['cercanos_hidrografia']:,}**"))
display(Markdown(f"- Siguen requiriendo revision manual: **{summary['requieren_revision_manual']:,}**"))
display(Markdown(f"- Probablemente excluibles por comercio simple: **{summary['probablemente_excluibles']:,}**"))

display(Markdown('### Total por localidad'))
display(summary['por_localidad'])

display(Markdown('### Conteo por categoria sugerida'))
display(summary['por_categoria_sugerida'])

display(Markdown('### Conteo por prioridad de revision'))
display(summary['por_prioridad_revision'])

## 4. Vista ordenada para auditar

La tabla se ordena por prioridad alta, menor distancia a hidrografia y categoria sugerida. Usa `query_maps` y `query_web` como texto para busquedas manuales; no hay scraping automatico.

In [ ]:
AUDIT_VIEW_COLUMNS = [
    'region_auditoria',
    'localidad_auditoria',
    'id',
    'nombre_de_la_unidad_economica',
    'codigo_de_la_clase_actividad_scian',
    'distancia_hidrografia_m',
    'rango_distancia_hidrografia',
    'keywords_detectadas',
    'categoria_sugerida',
    'prioridad_revision',
    'criterio_auditoria_sugerido',
    'query_maps',
    'query_web',
    'categoria_auditada',
    'notas_auditoria',
    'fuente_auditoria',
    'decision_estudio',
]

available_view_columns = [c for c in AUDIT_VIEW_COLUMNS if c in all_enriched.columns]

display(Markdown('### 20 registros con mayor prioridad'))
display(all_enriched[available_view_columns].head(20))

for region_id, df in enriched_tables.items():
    display(Markdown(f"### {REGION_METADATA[region_id]['label']}"))
    display(df[available_view_columns].head(30))

## 5. Visualizaciones simples

In [ ]:
plt.style.use('default')

category_by_region = (
    all_enriched
    .groupby(['localidad_auditoria', 'categoria_sugerida'])
    .size()
    .unstack(fill_value=0)
)
ax = category_by_region.plot(kind='bar', figsize=(9, 4))
ax.set_title('Categoria sugerida por localidad')
ax.set_xlabel('Localidad')
ax.set_ylabel('Registros')
ax.legend(title='Categoria sugerida')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

priority_by_region = (
    all_enriched
    .groupby(['localidad_auditoria', 'prioridad_revision'])
    .size()
    .unstack(fill_value=0)
)
ax = priority_by_region.plot(kind='bar', figsize=(9, 4))
ax.set_title('Prioridad de revision por localidad')
ax.set_xlabel('Localidad')
ax.set_ylabel('Registros')
ax.legend(title='Prioridad')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

distance = pd.to_numeric(all_enriched['distancia_hidrografia_auditoria_m'], errors='coerce').dropna()
if not distance.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(distance, bins=20, color='#4c78a8', edgecolor='white')
    axes[0].axvline(HYDRO_THRESHOLD_M, color='#b2182b', linestyle='--', label=f'{HYDRO_THRESHOLD_M} m')
    axes[0].set_title('Distancia a hidrografia')
    axes[0].set_xlabel('Metros')
    axes[0].set_ylabel('Registros')
    axes[0].legend()
    axes[1].boxplot(distance, vert=False)
    axes[1].axvline(HYDRO_THRESHOLD_M, color='#b2182b', linestyle='--')
    axes[1].set_title('Distribucion de distancia')
    axes[1].set_xlabel('Metros')
    plt.tight_layout()
    plt.show()
else:
    display(Markdown('No se encontro columna numerica de distancia a hidrografia.'))

## 6. Exportar archivos enriquecidos y reporte

Los archivos se guardan en `outputs/tables/auditoria_enriquecida/`. No se sobreescriben las plantillas originales.

In [ ]:
exported_paths = export_audit_outputs(
    enriched_tables,
    output_dir=ENRICHED_DIR,
    summary=summary,
)
report_path = write_markdown_report(
    path=ENRICHED_DIR / 'reporte_auditoria_revisar_denue.md',
    summary=summary,
    hydro_threshold_m=HYDRO_THRESHOLD_M,
)
exported_paths['reporte_md'] = report_path

display(Markdown('### Archivos generados'))
for key, path in exported_paths.items():
    display(Markdown(f'- `{path.relative_to(ROOT)}`'))

## 7. Validaciones

Estas validaciones revisan valores permitidos en `categoria_auditada`, conservacion de columnas originales, total de registros, presencia de las tres localidades y existencia de archivos exportados.

In [ ]:
validation = validate_audit_outputs(
    original_tables=templates,
    enriched_tables=enriched_tables,
    output_paths=exported_paths.values(),
    expected_total=EXPECTED_TOTAL_REVISAR,
)

display(validation)

if not validation['ok'].all():
    failing = validation.loc[~validation['ok']]
    raise ValueError(f'Hay validaciones pendientes:
{failing.to_string(index=False)}')

display(Markdown('### Validaciones completadas correctamente'))

## 8. Aplicar auditoria y generar mapas finales

Ejecuta esta celda solo despues de llenar manualmente `categoria_auditada` y `notas_auditoria` en las plantillas que consume el pipeline (`outputs/tables/auditoria_revisar_*.csv`). Los archivos enriquecidos son una ayuda de revision; los scripts 07/08 leen las plantillas originales.

In [ ]:
RUN_FINAL_SCRIPTS = False

if RUN_FINAL_SCRIPTS:
    subprocess.run([sys.executable, str(ROOT / 'scripts' / '07_apply_denue_audit.py')], cwd=ROOT, check=True)
    subprocess.run([sys.executable, str(ROOT / 'scripts' / '08_make_audited_maps.py')], cwd=ROOT, check=True)

    display(Markdown('### Outputs principales actualizados'))
    for path in [
        TABLES / 'denue_categorias_auditadas.csv',
        TABLES / 'resumen_denue_categorias_auditadas_por_localidad.csv',
        MAPS / '14_huejotzingo_denue_textil_auditado_alta_media.png',
        MAPS / '14_xalmimilulco_denue_textil_auditado_alta_media.png',
        MAPS / '14_san_martin_denue_textil_auditado_alta_media.png',
    ]:
        display(Markdown(f'- `{path.relative_to(ROOT)}`'))
else:
    display(Markdown('Scripts finales no ejecutados. Cambia `RUN_FINAL_SCRIPTS = True` cuando la auditoria manual este lista.'))